# 00 - REST API Summary (Read This First)

A complete overview of REST in one place. Each section explains the idea in plain words with an example. Open the numbered notebook only if you need code or more depth.

---

## 1. REST Basics
**What:** REST is a style for designing APIs. Everything is a **resource** (order, user, product) with its own **URL**, and **HTTP methods** say what to do with it. Defined by Roy Fielding in 2000.

**Example:** GET /orders/5 means "give me order 5".

**URL rules:** plural nouns, not verbs (/orders, not /getOrders); lowercase with hyphens (/pbc-reports); nest for relations (/orders/5/items).

**Six constraints:** client-server (frontend and backend separate), stateless (no memory between requests), cacheable, uniform interface (same pattern everywhere), layered (proxies/load balancers in between are invisible), code on demand (optional).

**REST vs RESTful:** REST is the style; RESTful is an API that follows it.

---

## 2. HTTP Methods and Idempotency
**What:** The method tells the server the action.
- GET = read, POST = create, PUT = replace whole resource, PATCH = change some fields, DELETE = remove
- PUT vs PATCH: PUT without a field can wipe it; PATCH touches only what you send

**Idempotency:** calling once or ten times leaves the same result. GET, PUT, DELETE are idempotent. POST is not, because each call creates a new record.

**Problem:** network drops, user clicks Pay again → charged twice.
**Fix:** client sends an **Idempotency-Key** header; server saves the result for that key and returns it on repeats. Razorpay and Stripe do this.

---

## 3. Status Codes
**What:** a 3-digit number telling the client how the request went.
- **2xx success:** 200 OK, 201 Created (after POST), 204 No Content (after DELETE)
- **3xx redirect:** 301 moved permanently, 304 use your cached copy
- **4xx client mistake:** 400 bad input, 401 not logged in, 403 logged in but not allowed, 404 not found, 409 conflict (e.g. out of stock), 422 validation failed (FastAPI default), 429 too many requests
- **5xx server mistake:** 500 bug, 502 upstream service failed, 503 overloaded/maintenance

**Remember:** 401 = "Who are you?", 403 = "I know you, but no."

---

## 4. Request and Response Design
**Four places data travels:**
- **Path:** which one → /orders/5
- **Query:** which ones (filter, sort, page) → /orders?status=paid
- **Body:** the data to create/update → {"address": "Hyderabad"}
- **Headers:** metadata → Authorization token, Content-Type, Idempotency-Key

**Responses:** consistent JSON shape, return the resource with its id after writes, never expose sensitive fields like password_hash (use a response model), and wrap lists with total, page, size.

---

## 5. Pagination, Filtering, Sorting
**Why:** never return a million rows at once.

- **Offset pagination:** client sends page and size; skip = (page − 1) × size. Simple, can jump pages, but slow on huge tables and can repeat items when data changes. Good for admin tables.
- **Cursor pagination:** client sends the last id seen; WHERE id > last_id. Fast and consistent, but no page jumping. Good for feeds and big data.
- **Filtering/sorting:** in query params, e.g. ?status=paid&sort=-date (minus = descending). Whitelist sortable fields to avoid SQL injection. Order: filter → sort → paginate.
- **Always cap page size** (e.g. max 100).

---

## 6. Versioning
**What:** lets the API change without breaking existing clients.
- Put the version in the URL: /v1/orders, /v2/orders
- New version only for **breaking** changes: removing/renaming a field, changing a type or shape
- Adding an optional field or new endpoint is not breaking
- Keep v1 running until old clients (like old mobile app versions) migrate

---

## 7. Authentication and JWT
**Authentication** = who are you (fails with 401). **Authorization** = what can you do (fails with 403).

**JWT:** a signed token given after login, sent as Authorization: Bearer <token>.
- Three parts: header (algorithm), payload (user id, role, expiry), signature (proves it wasn't changed)
- Payload is **encoded, not encrypted**: anyone can read it, nobody can alter it. Never store secrets in it.

**Access vs refresh token:** access token is short-lived (≈15 min) and sent on every request; refresh token is long-lived and only used to get a new access token. Logout revokes the refresh token.

**My project:** Azure Entra ID issues the JWT; our backend only verifies signature, expiry and roles.

---

## 8. Error Handling
- Every error has the **same JSON shape**: code (for the frontend), message (for users), details
- Use the right status code with it
- Define custom exceptions (NotFound, OutOfStock) and map them to status codes in **one central handler**
- On a 500, never show the stack trace; log it server-side with a request id and return only that id

---

## 9. Statelessness, Caching, Rate Limiting
- **Stateless:** server stores no session; the JWT travels with each request, so any server behind a load balancer can handle it. This is what makes scaling easy.
- **Caching (cache-aside):** check Redis → hit: return; miss: read DB, store with a **TTL**, return. Delete the cache entry when data changes. Cache product lists, not stock counts or payments.
- **Rate limiting:** cap requests per user/IP (e.g. 5 login attempts/minute). Over the limit → **429** with Retry-After. Keep counters in Redis so all servers share them.

---

## 10. CORS
**What:** a **browser-only** rule that stops a page from one origin reading responses from another origin unless the server allows it. Origin = protocol + domain + port.

**How:** the browser sends an Origin header; the backend checks it against allow_origins.
- "*" means any website can call your API from a browser → only for local testing
- Production: list real frontend domains only
- Postman and server-to-server calls ignore CORS
- **CORS is not security on its own:** CORS decides which site, JWT decides which user
- A same-domain proxy (Nginx forwarding /api to backend) avoids CORS entirely

---

## 11. REST vs GraphQL vs gRPC
- **REST:** many URLs, JSON. Simple and cacheable, but over-fetches (too many fields) or under-fetches (needs several calls). Best for public APIs.
- **GraphQL:** one endpoint; client asks for exactly the fields it wants, including related data, in one call. Harder to cache and secure. Best for mobile apps and complex screens.
- **gRPC:** binary Protocol Buffers over HTTP/2 with a strict .proto contract. Very fast but not human-readable and weak in browsers. Best for internal service-to-service calls.

---

## 12. Webhooks and Real-Time
- **Webhook:** another service calls **your** URL when something happens ("don't call us, we'll call you"). Example: Razorpay POSTs "payment success" to /webhooks/razorpay.
  - Verify the **HMAC signature** with the shared secret
  - Skip duplicate event ids (they retry)
  - Return 200 fast; do heavy work in the background
- **Polling:** client keeps asking; wasteful and slow
- **SSE:** one open connection, server streams updates one way. Used for LLM streaming and my LangGraph node-progress UI
- **WebSocket:** two-way live connection, e.g. chat apps
- **HATEOAS:** responses include links to next actions; rarely used